# Inspect Defense Artifacts

This notebook inspects defense-input artifacts saved by `--save-defense-inputs`. It uses memory mapping for the SHAP matrix so the whole file is not loaded into RAM.


In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACT_RELATIVE = Path("results/ember2024/win64/random-defense/attack_artifacts/ember2024_win64__lightgbm__combined_shap__combined_shap__feasible")
ARTIFACT_DIR = PROJECT_ROOT / ARTIFACT_RELATIVE

print("Project root:", PROJECT_ROOT)
print("Artifact dir:", ARTIFACT_DIR)
print("Exists:", ARTIFACT_DIR.exists())


Project root: /Users/falcon/Machine Learning/Defense GMM-Maha
Artifact dir: /Users/falcon/Machine Learning/Defense GMM-Maha/results/ember2024/win64/random-defense/attack_artifacts/ember2024_win64__lightgbm__combined_shap__combined_shap__feasible
Exists: True


In [5]:
artifact_root = PROJECT_ROOT / "results/ember2024/win64/random-defense/attack_artifacts"
available_artifacts = sorted([p for p in artifact_root.glob("*") if p.is_dir()]) if artifact_root.exists() else []

print("Available artifact folders:")
for i, path in enumerate(available_artifacts):
    marker = "<-- selected" if path == ARTIFACT_DIR else ""
    print(f"{i}: {path.name} {marker}")

if not ARTIFACT_DIR.exists():
    if available_artifacts:
        ARTIFACT_DIR = available_artifacts[0]
        print("\nDefault artifact folder was missing, using first available folder:")
        print(ARTIFACT_DIR)
    else:
        raise FileNotFoundError(f"No artifact folders found under {artifact_root}")


Available artifact folders:
0: ember2024_win64__lightgbm__combined_shap__combined_shap__feasible <-- selected
1: ember2024_win64__lightgbm__shap_largest_abs__argmin_Nv_sum_abs_shap__feasible 
2: ember2024_win64__lightgbm__shap_largest_abs__min_population_new__feasible 


In [6]:
for path in sorted(ARTIFACT_DIR.glob("*")):
    print(f"{path.name:45s} {path.stat().st_size / 1024 / 1024:10.2f} MB")


backdoored_model_benign_shap.npy                 1018.80 MB
backdoored_model_benign_shap_base_value.npy         0.40 MB
defense_metadata.json                               0.00 MB
defense_metadata.npz                                1.43 MB


In [7]:
with (ARTIFACT_DIR / "defense_metadata.json").open("r", encoding="utf-8") as f:
    metadata_json = json.load(f)

metadata_json


{'dataset': 'ember2024_win64',
 'defense_shap_base_value_file': 'backdoored_model_benign_shap_base_value.npy',
 'defense_shap_file': 'backdoored_model_benign_shap.npy',
 'defense_shap_saved': True,
 'index_meaning': {'benign_watermarked_idx': 'benign-labeled row ids in watermarked_X.npy / y_train_watermarked order',
  'poison_mask_benign': 'boolean mask aligned to benign_watermarked_idx',
  'poisoned_original_idx': 'row ids in the pre-poisoning X_train array used by this run',
  'poisoned_source_idx': 'source dataset row ids when available, otherwise equal to poisoned_original_idx',
  'poisoned_watermarked_idx': 'row ids in watermarked_X.npy / y_train_watermarked order'},
 'metadata_file': 'defense_metadata.npz',
 'model_id': 'lightgbm',
 'num_benign_rows': 104000,
 'num_malware_rows': 104000,
 'num_poisoned_benign_rows': 1040,
 'num_poisoned_rows': 1040,
 'num_train_rows': 208000,
 'row_order': 'watermarked_X order is malware rows, then clean benign rows, then poisoned benign rows',
 

In [8]:
shap = np.load(ARTIFACT_DIR / "backdoored_model_benign_shap.npy", mmap_mode="r")
base_value = np.load(ARTIFACT_DIR / "backdoored_model_benign_shap_base_value.npy", mmap_mode="r")
meta = np.load(ARTIFACT_DIR / "defense_metadata.npz")

print("SHAP shape:", shap.shape, "dtype:", shap.dtype)
print("Base value shape:", base_value.shape, "dtype:", base_value.dtype)
print("Metadata keys:")
for key in meta.files:
    arr = meta[key]
    print(f"  {key:28s} shape={arr.shape!s:14s} dtype={arr.dtype}")


SHAP shape: (104000, 2568) dtype: float32
Base value shape: (104000,) dtype: float32
Metadata keys:
  watermarked_original_idx     shape=(208000,)      dtype=int64
  watermarked_source_idx       shape=(208000,)      dtype=int64
  train_gw_to_be_watermarked   shape=(1040,)        dtype=int64
  poisoned_original_idx        shape=(1040,)        dtype=int64
  poisoned_source_idx          shape=(1040,)        dtype=int64
  poisoned_watermarked_idx     shape=(1040,)        dtype=int64
  benign_watermarked_idx       shape=(104000,)      dtype=int64
  benign_original_idx          shape=(104000,)      dtype=int64
  benign_source_idx            shape=(104000,)      dtype=int64
  poison_mask_full             shape=(208000,)      dtype=bool
  poison_mask_benign           shape=(104000,)      dtype=bool
  test_mw_to_be_watermarked    shape=(118192,)      dtype=int64
  y_train_watermarked          shape=(208000,)      dtype=int32


In [9]:
summary = {
    "num_train_rows": int(meta["y_train_watermarked"].shape[0]),
    "num_benign_rows": int(meta["benign_watermarked_idx"].shape[0]),
    "num_poisoned_rows": int(meta["poison_mask_full"].sum()),
    "num_poisoned_benign_rows": int(meta["poison_mask_benign"].sum()),
    "shap_rows_match_benign_rows": bool(shap.shape[0] == meta["benign_watermarked_idx"].shape[0]),
    "poison_mask_benign_rows_match": bool(meta["poison_mask_benign"].shape[0] == meta["benign_watermarked_idx"].shape[0]),
}
pd.Series(summary)


num_train_rows                   208000
num_benign_rows                  104000
num_poisoned_rows                  1040
num_poisoned_benign_rows           1040
shap_rows_match_benign_rows        True
poison_mask_benign_rows_match      True
dtype: object

In [10]:
preview_keys = [
    "benign_watermarked_idx",
    "poison_mask_benign",
    "poisoned_watermarked_idx",
    "poisoned_original_idx",
    "poisoned_source_idx",
]

for key in preview_keys:
    print("\n", key)
    print(meta[key][:20])



 benign_watermarked_idx
[104000 104001 104002 104003 104004 104005 104006 104007 104008 104009
 104010 104011 104012 104013 104014 104015 104016 104017 104018 104019]

 poison_mask_benign
[False False False False False False False False False False False False
 False False False False False False False False]

 poisoned_watermarked_idx
[206960 206961 206962 206963 206964 206965 206966 206967 206968 206969
 206970 206971 206972 206973 206974 206975 206976 206977 206978 206979]

 poisoned_original_idx
[131711  84583 160365  40368 130942 198418 189741  62734 161749  55678
 151200  53060  13594  95792  94494   6537 169517  16526  52632 194222]

 poisoned_source_idx
[656927 421694 801443 201834 653177 992134 948958 313695 808136 278562
 754670 265727  68188 476919 470637  32507 847595  83192 263425 971365]


In [11]:
row_ids = [0, 1, 2]
col_ids = list(range(10))
pd.DataFrame(
    shap[row_ids][:, col_ids],
    index=[f"benign_shap_row_{i}" for i in row_ids],
    columns=[f"feature_{j}" for j in col_ids],
)


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9
benign_shap_row_0,0.006050,0.0,0.0,0.0,0.0,-0.000022,-0.002775,-0.077393,-0.006140,0.0
benign_shap_row_1,-0.000692,0.0,0.0,0.0,0.0,0.000055,-0.003084,-0.036615,-0.000725,0.0
benign_shap_row_2,0.004706,0.0,0.0,0.0,0.0,0.000151,-0.002986,-0.084695,-0.024200,0.0


In [12]:
poisoned_benign_positions = np.flatnonzero(meta["poison_mask_benign"])
print("First poisoned positions inside benign-only SHAP matrix:", poisoned_benign_positions[:20])

if poisoned_benign_positions.size:
    first_poison_pos = int(poisoned_benign_positions[0])
    print("First poisoned benign SHAP row index:", first_poison_pos)
    print("First 10 SHAP values:", shap[first_poison_pos, :10])


First poisoned positions inside benign-only SHAP matrix: [102960 102961 102962 102963 102964 102965 102966 102967 102968 102969
 102970 102971 102972 102973 102974 102975 102976 102977 102978 102979]
First poisoned benign SHAP row index: 102960
First 10 SHAP values: [-1.8325663e-01  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  3.1763120e-05 -2.4167390e-03 -3.1524587e-02
 -9.1510412e-04  0.0000000e+00]
